# Chapter 3 — Exception Levels and Privilege Model

## Concept
AArch64 defines four Exception Levels (EL0–EL3). EL3 is the most privileged
(Secure Monitor), EL2 is the Hypervisor, EL1 is the OS kernel, EL0 is
user space. Key system registers: SCR_EL3 (controls EL3 routing),
HCR_EL2 (hypervisor config), VBAR_EL1/EL2/EL3 (vector base addresses).
PSCI calls use SMC (EL3 target) or HVC (EL2 target) instructions.

## Lab Objectives
1. Capture the QEMU serial boot log during system bring-up.
2. Identify EL2 entry, UEFI hand-off, and EL1 Linux entry in the log.
3. Assert PSCI CPU_ON call is visible in the trace.
4. Confirm no EL3 fault in the boot log.

> **QEMU Fidelity Note** — Results from this lab are functionally valid on
> QEMU `virt` machine with HVF (macOS Apple Silicon). Behaviours that differ
> from real Neoverse silicon are annotated inline. See Section 6 of the
> project plan for the full gap table.


In [ ]:
import sys, os, pathlib, time
from IPython.display import display, HTML

# ── USER CONFIGURATION — edit these paths before running ──────────────────────
LABS_ROOT    = pathlib.Path.home() / "arm_qemu_labs"
SHARED_DIR   = LABS_ROOT / "shared"
DISK_IMAGE   = LABS_ROOT / "images" / "ubuntu-24.04-arm64.qcow2"
SEED_ISO     = LABS_ROOT / "images" / "seed.iso"   # cloud-init seed
FIRMWARE     = LABS_ROOT / "firmware" / "QEMU_EFI.fd"
CONSOLE_USER = "ubuntu"
CONSOLE_PASS = "arm-lab-2026"
CPU          = "cortex-a76"
RAM          = "2G"
SMP          = 1
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(SHARED_DIR))
from qemu_launcher  import QEMULauncher
from qmp_client     import QMPClient
from serial_console import SerialConsole
from assert_lib     import (assert_true, assert_false, assert_equal,
                             assert_contains, assert_not_contains,
                             assert_qmp_ok, assert_in_range, summary, reset)
reset()

import shutil
assert shutil.which("qemu-system-aarch64"), (
    "FATAL: qemu-system-aarch64 not found in PATH. Run setup_qemu_labs.sh.")
assert FIRMWARE.exists(),   f"FATAL: Firmware not found at {FIRMWARE}"
assert DISK_IMAGE.exists(), f"FATAL: Disk image not found at {DISK_IMAGE}"
assert SEED_ISO.exists(),   f"FATAL: Seed ISO not found at {SEED_ISO}"
print("✓ Environment check passed")
print(f"  qemu-system-aarch64 : {shutil.which('qemu-system-aarch64')}")
print(f"  Firmware            : {FIRMWARE}")
print(f"  Disk image          : {DISK_IMAGE}")


In [ ]:
launcher = QEMULauncher(
    cpu=CPU, ram=RAM, smp=SMP,
    firmware=str(FIRMWARE),
    disk_image=str(DISK_IMAGE),
    seed_iso=str(SEED_ISO),
    extra_args=[],
)
launcher.launch()
launcher.wait_ready(timeout=15)
print(f"QEMU running — QMP port {launcher.qmp_port}, serial port {launcher.serial_port}")


In [ ]:
qmp = QMPClient(port=launcher.qmp_port)
qmp.connect(timeout=30)

sc = SerialConsole(port=launcher.serial_port)
sc.connect(timeout=30)

print("Waiting for guest boot (up to 120 s on HVF, longer on TCG)…")
sc.wait_for_boot(timeout=180)
sc.login(CONSOLE_USER, CONSOLE_PASS)
print("Guest ready.")


In [ ]:
# ── Step 1: Capture raw serial boot log from QEMU log file ───────────────────
# The boot log was written to launcher.log_file during wait_for_boot().
with open(launcher.log_file, "r", errors="replace") as fh:
    full_log = fh.read()
print(f"Log file: {launcher.log_file}")
print(f"Log size: {len(full_log)} chars")
print("\n── First 1500 chars of boot log ──")
print(full_log[:1500])


In [ ]:
# ── Step 2: Search for EL2 / UEFI / kernel entry markers ─────────────────────
import re

# UEFI QEMU firmware prints "UEFI" or "EDK II"
uefi_markers = re.findall(r"(EDK II|UEFI|Booting|BDS)", full_log)
print(f"UEFI/BDS markers found: {uefi_markers[:10]}")

# Linux kernel AArch64 entry: "Booting Linux on physical CPU" or "Linux version"
kernel_entry = re.search(r"Linux version \S+", full_log)
if kernel_entry:
    print(f"\nKernel entry: {kernel_entry.group()}")
else:
    print("Kernel entry marker not in QEMU stdout log (may be on serial only)")
    # Fall back to dmesg from guest
    kernel_ver = sc.run_command("uname -r", timeout=10)
    print(f"Guest kernel version: {kernel_ver}")


In [ ]:
# ── Step 3: Read dmesg — confirm EL1 boot and PSCI ───────────────────────────
dmesg_out = sc.run_command("dmesg", timeout=20)
print(dmesg_out[:3000])


In [ ]:
# ── Step 4: Grep for EL / PSCI / exception keywords ─────────────────────────
psci_lines = sc.run_command("dmesg | grep -i 'psci\|el2\|el1\|exception'", timeout=10)
print("PSCI / EL lines in dmesg:\n", psci_lines)


In [ ]:
# ── Step 5: Check for EL3 fault markers (should be absent) ──────────────────
fault_lines = sc.run_command(
    "dmesg | grep -iE 'EL3|synchronous abort|prefetch abort|unhandled fault'",
    timeout=10
)
print("Fault lines (expected empty):", repr(fault_lines))


In [ ]:
# ── Step 6: Verify kernel booted in EL1 via sysfs (if available) ─────────────
# ARM64 Linux exposes some EL info via /sys/devices/system/cpu
cpu0_caps = sc.run_command(
    "ls /sys/devices/system/cpu/cpu0/ 2>/dev/null || echo 'not present'",
    timeout=10
)
print(cpu0_caps)


In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert_true(len(full_log) > 100,
            "QEMU log file is non-empty",
            detail=f"{len(full_log)} chars captured",
            action="Check launcher.log_file path; QEMU may have exited early")

assert_contains(dmesg_out, r"Linux version",
                "dmesg contains 'Linux version' string",
                action="Guest kernel may not have booted; check serial output")

assert_contains(dmesg_out, r"Booting Linux on physical CPU",
                "dmesg shows AArch64 physical CPU boot message",
                action="Check -cpu flag and machine type")

# PSCI: Linux registers PSCI during SMP bringup; visible even on 1 CPU
assert_contains(dmesg_out, r"[Pp][Ss][Cc][Ii]",
                "dmesg contains PSCI reference",
                action="Kernel may lack PSCI support; check CONFIG_ARM_PSCI_FW")

assert_contains(dmesg_out, r"EL2|el2|Hyp|hyp",
                "dmesg references EL2 / Hyp mode",
                action="Kernel may have booted without EL2 — check -machine virt")

# No EL3 fault
assert_not_contains(
    dmesg_out,
    r"Unhandled fault.*EL3|EL3.*abort",
    "No EL3 fault in dmesg",
    action="EL3 fault present — check firmware and SCR_EL3 configuration",
)

assert_not_contains(
    fault_lines,
    r"synchronous abort|prefetch abort",
    "No synchronous / prefetch abort in boot log",
    action="Inspect dmesg for fault address and registers",
)


In [ ]:
# ── Teardown: always runs even if earlier cells raised ────────────────────────
try:
    qmp.quit()
except Exception:
    pass
try:
    sc.close()
except Exception:
    pass
try:
    launcher.terminate()
except Exception:
    pass
print("Teardown complete. QEMU process terminated.")


## Lab Summary

| Assertion | Expected | Notes |
|-----------|----------|-------|
| QEMU log non-empty | Yes | QEMU writing stdout |
| dmesg: Linux version | Present | Kernel booted |
| dmesg: AArch64 CPU boot | Present | ARM64 boot path |
| dmesg: PSCI reference | Present | PSCI driver registered |
| dmesg: EL2 / Hyp reference | Present | Kernel aware of EL2 |
| No EL3 fault | Absent | Clean boot |
| No synchronous abort | Absent | No exception during boot |

> **Fidelity note**: QEMU `virt` machine does not load real TF-A (EL3 firmware)
> unless `bl31.bin` is explicitly provided. The UEFI stub (QEMU_EFI.fd) starts
> in EL2. On real Neoverse, TF-A runs in EL3 and the SCR_EL3 routing differs.

## Next Steps
→ **Chapter 4**: GIC — inject SPI via QMP and observe the handler counter increment.
